# Holdout (2024-01-02 to 2026-08-31)

Run once, after everything in `03_evaluation.ipynb` is final.
Nothing may be changed in response to what this shows.

Success criterion 3: the holdout mean IC has the same sign as development, with |NW t| >= 2.

In [1]:
import numpy as np
import pandas as pd

from shortvol import evaluate as ev, features, finra, prices

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
V0 = features.estimate_v0(features.add_all(panel, days))["v0"]  # 2016-2019 short volume only
df = features.add_all(panel, days, v0=V0)

SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume"]
TARGETS = ["fwd_oc", "fwd_cc"]
for s in SIGNALS:
    df[s + "_n"] = ev.neutralize(df, s, CONTROLS)
df["size_tercile"] = ev.terciles(df, "log_adv20", ["small", "mid", "large"])
df["offex_tercile"] = ev.terciles(df, "off_exchange_share", ["low", "mid", "high"])

In [2]:
d = df[df.date >= "2024-01-02"]
print(f"holdout stock-days: {len(d):,}, days: {d.date.nunique()}")

holdout stock-days: 335,252, days: 668


In [3]:
rows = []
for s in SIGNALS:
    for t in TARGETS:
        for kind, col in [("raw", s), ("neutralized", s + "_n")]:
            rows.append({"signal": s, "target": t, "version": kind, **ev.summarize_ic(ev.daily_ic(d, col, t))})
headline = pd.DataFrame(rows).set_index(["signal", "target", "version"])
headline

mean_ic    nw_t  ic_ir_annual  pct_positive  days
signal       target version                                                       
svr5         fwd_oc raw          -0.0043 -1.2264       -0.7742        0.4933   667
                    neutralized  -0.0011 -0.4009       -0.2764        0.4963   667
             fwd_cc raw          -0.0077 -2.2455       -1.3779        0.4550   666
                    neutralized  -0.0026 -1.0697       -0.6874        0.4760   666
svr1         fwd_oc raw          -0.0086 -2.5705       -1.6688        0.4573   667
                    neutralized  -0.0065 -2.4373       -1.7382        0.4483   667
             fwd_cc raw          -0.0085 -2.6886       -1.6334        0.4520   666
                    neutralized  -0.0039 -1.7408       -1.0856        0.4610   666
svr_abnormal fwd_oc raw          -0.0020 -0.7530       -0.4896        0.5127   667
                    neutralized  -0.0001 -0.0684       -0.0472        0.5202   667
             fwd_cc raw          -0.0014 -0.5486       -0.3431        0.5090   666
                    neutralized   0.0003  0.1263        0.0814        0.5120   666
svr5_shrunk  fwd_oc raw          -0.0043 -1.2134       -0.7650        0.4918   667
                    neutralized  -0.0010 -0.3918       -0.2699        0.4993   667
             fwd_cc raw          -0.0079 -2.2748       -1.3994        0.4595   666
                    neutralized  -0.0028 -1.1091       -0.7150        0.4775   666

In [4]:
dev = ev.summarize_ic(ev.daily_ic(df[df.date <= "2023-12-29"], "svr5_n", "fwd_oc"))
hold = headline.loc[("svr5", "fwd_oc", "neutralized")]
c3 = np.sign(hold.mean_ic) == np.sign(dev["mean_ic"]) and abs(hold.nw_t) >= 2
print(f"development mean IC {dev['mean_ic']:.4f}; holdout mean IC {hold.mean_ic:.4f}, NW t {hold.nw_t:.2f}")
print(f"3. {'PASS' if c3 else 'FAIL'}")

development mean IC 0.0003; holdout mean IC -0.0011, NW t -0.40
3. FAIL


In [5]:
pd.concat({
    "size": ev.ic_by_group(d, "svr5_n", "fwd_oc", "size_tercile"),
    "off-exchange share": ev.ic_by_group(d, "svr5_n", "fwd_oc", "offex_tercile"),
})

mean_ic    nw_t  ic_ir_annual  pct_positive     days
size               large  -0.0046 -1.2226       -0.8056        0.4648 667.0000
                   mid     0.0018  0.4737        0.3038        0.4708 667.0000
                   small   0.0013  0.3492        0.2304        0.5217 667.0000
off-exchange share high   -0.0008 -0.2384       -0.1531        0.5097 667.0000
                   low    -0.0013 -0.3758       -0.2324        0.4948 667.0000
                   mid    -0.0004 -0.1134       -0.0774        0.4963 667.0000

## Quintile long-short (same construction as 03)
Long the top quintile of each neutralized signal, short the bottom quintile, equal weight, rebalanced daily.

In [6]:
pd.DataFrame({f"{s}_n": ev.quintile_spread(d, s + "_n", "fwd_oc") for s in SIGNALS}).T

,spread_bps,spread_nw_t,turnover,breakeven_bps
svr5_n,1.2115,0.9862,0.6846,1.7697
svr1_n,-1.8636,-1.4852,1.8764,0.9931
svr_abnormal_n,0.2095,0.2085,0.8186,0.2559
svr5_shrunk_n,1.3690,1.0988,0.6788,2.0168
